# ETF AI ハンズオン
## Part 1: データ探索 & Cortex AI Functions

このノートブックでは、ポートフォリオマネージャーの業務を想定した **データ探索** と
**Snowflake Cortex AI Functions** の活用方法を学びます。

### このパートで体験できること

1. **データ確認**: ETFポートフォリオデータの構造を把握する
2. **AI_COMPLETE**: マーケットニュースを指定モデルで自動要約する
3. **AI_SENTIMENT**: ニュースのセンチメント（強気/弱気）を数値化する
4. **AI_CLASSIFY**: ニュースを投資判断カテゴリに自動分類する
5. **AI_EXTRACT**: 月次レポート画像から重要情報を構造化して抽出する
6. **AI_AGG**: 全ニュースを横断した市場サマリーを自動生成する

### 🚀 体験ポイント

> **「毎朝の情報収集が10分から10秒に！」**
>
> 40本のニュース記事を読んで市場動向を把握する作業が、
> SQLを1行書くだけで完了します。

### 前提条件
- `setup.sql` の実行が完了していること
- ウェアハウス: `DEMO_WH`
- データベース: `ETF_AI_HANDSON_DB`

> ⏱️ **このパートの目安時間: 35分**


In [ ]:
USE DATABASE ETF_AI_HANDSON_DB;
USE SCHEMA DEMO_SCHEMA;
USE WAREHOUSE DEMO_WH;
SELECT
    CURRENT_DATABASE()  AS DB,
    CURRENT_SCHEMA()    AS SCHEMA,
    CURRENT_WAREHOUSE() AS WH,
    CURRENT_ROLE()      AS ROLE;

## 1. データ探索

まず、setup.sql で作成されたデータの構造を確認します。

> ⏱️ **目安: 10分**

In [ ]:
-- テーブル一覧とレコード数を確認
SELECT
    TABLE_NAME  AS "テーブル名",
    ROW_COUNT   AS "レコード数",
    CASE TABLE_NAME
        WHEN 'DIM_ETF'          THEN 'ETF銘柄マスタ（10銘柄）'
        WHEN 'DIM_PORTFOLIO'    THEN 'ポートフォリオマスタ（10本）'
        WHEN 'FACT_HOLDINGS'    THEN '保有ETF明細（最新保有状況）'
        WHEN 'FACT_PERFORMANCE' THEN '月次パフォーマンス実績（24ヶ月分）'
        WHEN 'FACT_REBALANCING' THEN 'リバランス履歴（27件）'
        WHEN 'MARKET_NEWS'      THEN 'マーケットニュース（40件）'
        WHEN 'ETF_DESCRIPTIONS' THEN 'ETFファンド説明文書（Cortex Search用）'
        ELSE ''
    END AS "説明"
FROM INFORMATION_SCHEMA.TABLES
WHERE TABLE_SCHEMA = 'DEMO_SCHEMA'
  AND TABLE_TYPE = 'BASE TABLE'
ORDER BY TABLE_NAME;


In [ ]:
-- ETF銘柄一覧（ラインナップ一覧）
SELECT
    ETF_CODE                AS "コード",
    ETF_NAME_SHORT          AS "略称",
    CATEGORY                AS "カテゴリ",
    ASSET_CLASS             AS "資産クラス",
    TO_CHAR(AUM_BILLION_JPY, '9,999.99') || '億円' AS "純資産",
    TO_CHAR(NAV, '999,999') || '円'  AS "基準価額(100口)",
    TRIM(TO_CHAR(EXPENSE_RATIO * 100, '90.0000')) || '%' AS "コスト(年率)",
    CASE WHEN NISA_ELIGIBLE THEN '○' ELSE '-' END AS "NISA"
FROM DIM_ETF
ORDER BY AUM_BILLION_JPY DESC;


In [ ]:
-- ポートフォリオ一覧と運用概要
SELECT
    PORTFOLIO_ID    AS "ID",
    PORTFOLIO_NAME  AS "ポートフォリオ名",
    MANAGER_NAME    AS "PM",
    RISK_PROFILE    AS "リスクプロファイル",
    TARGET_RETURN   || '%' AS "目標リターン(年率)",
    TO_CHAR(AUM_MILLION_JPY / 100, '99,999.0') || '億円' AS "運用資産"
FROM DIM_PORTFOLIO
ORDER BY AUM_MILLION_JPY DESC;

In [ ]:
-- ポートフォリオ保有明細（P004: グローバル分散成長の例）
SELECT
    h.PORTFOLIO_ID   AS "ポートフォリオ",
    e.ETF_NAME_SHORT AS "ETF名称",
    e.ETF_CODE       AS "コード",
    e.CATEGORY       AS "カテゴリ",
    TO_CHAR(h.WEIGHT_PCT, '99.999') || '%'  AS "比重",
    TO_CHAR(h.MARKET_VALUE_JPY / 100000000, '99,999.0') || '億円' AS "時価評価額",
    TO_CHAR(h.UNREALIZED_GAIN_JPY / 100000000, '9,999.0') || '億円' AS "含み損益",
    TO_CHAR(h.UNREALIZED_GAIN_PCT, '99.0') || '%' AS "含み損益率"
FROM FACT_HOLDINGS h
JOIN DIM_ETF e ON h.ETF_CODE = e.ETF_CODE
WHERE h.PORTFOLIO_ID = 'P004'
ORDER BY h.WEIGHT_PCT DESC;

In [ ]:
-- 各ポートフォリオの2025年通年パフォーマンス比較
SELECT
    p.PORTFOLIO_ID    AS "ID",
    dp.PORTFOLIO_NAME AS "ポートフォリオ名",
    TO_CHAR(p.RETURN_1Y_PCT,  '99.0') || '%' AS "1年リターン",
    TO_CHAR(p.VOLATILITY_1Y,  '99.0') || '%' AS "ボラティリティ",
    TO_CHAR(p.SHARPE_RATIO,   '9.00')        AS "シャープレシオ",
    TO_CHAR(p.MAX_DRAWDOWN,   '99.0') || '%' AS "最大ドローダウン"
FROM FACT_PERFORMANCE p
JOIN DIM_PORTFOLIO dp ON p.PORTFOLIO_ID = dp.PORTFOLIO_ID
WHERE p.PERF_DATE = '2025-12-31'
  AND p.ETF_CODE IS NULL
ORDER BY p.RETURN_1Y_PCT DESC;

## 2. Cortex AI Functions の活用

### Cortex AI Functions とは？

Snowflake Cortex AI Functions は、SQL の中から直接呼び出せる AI 機能です。
ローカルへの AI モデルのデプロイや API キーの管理は **一切不要** です。

| 関数 | 用途 | 活用例 |
|---|---|---|
| `AI_COMPLETE` | LLM呼び出し・要約 | 任意モデルでニュースを1〜2文に圧縮 |
| `CORTEX.SENTIMENT` | 感情分析 | 強気/弱気トーンを -1〜+1 で数値化 |
| `AI_CLASSIFY` | テキスト分類 | 買い材料/様子見/売り材料の自動判定 |
| `AI_EXTRACT` | 情報抽出 | 月次レポート画像から数値・銘柄を構造化 |
| `AI_AGG` | 集約質問応答 | 全ニュースからの市場サマリー自動生成 |

> ⏱️ **目安: 25分**


### 2-1. AI_COMPLETE（ニュース自動要約）

`AI_COMPLETE(model, prompt)` で指定した LLM モデルを呼び出し、ニュース本文を要約します。
モデル名を変えるだけで GPT-4o・Claude・Mistral など複数のモデルを比較できます。

毎朝の情報収集作業を大幅に効率化できます。


In [ ]:
-- ニュース本文を自動要約（1〜2文に圧縮）
SELECT
    NEWS_DATE   AS "日付",
    HEADLINE    AS "ヘッドライン",
    AI_COMPLETE(
        'claude-sonnet-4-6',
        '以下のニュース本文を1〜2文で簡潔に日本語で要約してください:\n\n' || BODY
    ) AS "AI要約（自動生成）"
FROM MARKET_NEWS
ORDER BY NEWS_DATE DESC
LIMIT 5;

### 2-2. SNOWFLAKE.CORTEX.SENTIMENT（センチメント分析）

`SNOWFLAKE.CORTEX.SENTIMENT(text)` でテキストのポジティブ/ネガティブを数値で返します。
- **+1.0 に近い**: 強いポジティブ（買い材料）
- **-1.0 に近い**: 強いネガティブ（売り材料）
- **0 に近い**: ニュートラル


In [ ]:
-- ニュースのセンチメント分析（全記事スコアリング）
SELECT
    NEWS_DATE   AS "日付",
    HEADLINE    AS "ヘッドライン",
    ROUND(SNOWFLAKE.CORTEX.SENTIMENT(BODY), 2) AS "スコア(-1〜+1)",
    CASE
        WHEN SNOWFLAKE.CORTEX.SENTIMENT(BODY) >= 0.1  THEN 'ポジティブ (買い材料)'
        WHEN SNOWFLAKE.CORTEX.SENTIMENT(BODY) <= -0.1 THEN 'ネガティブ (売り材料)'
        ELSE 'ニュートラル (様子見)'
    END AS "判定",
    RELEVANT_ETFS AS "関連ETF"
FROM MARKET_NEWS
ORDER BY SNOWFLAKE.CORTEX.SENTIMENT(BODY) DESC;


In [ ]:
-- センチメント分析結果をテーブルに永続化
UPDATE MARKET_NEWS
SET SENTIMENT = CASE
    WHEN SNOWFLAKE.CORTEX.SENTIMENT(BODY) >= 0.1  THEN 'ポジティブ'
    WHEN SNOWFLAKE.CORTEX.SENTIMENT(BODY) <= -0.1 THEN 'ネガティブ'
    ELSE 'ニュートラル'
END;

-- 結果確認
SELECT
    SENTIMENT       AS "センチメント",
    COUNT(*)        AS "件数",
    LISTAGG(NEWS_ID, ', ') WITHIN GROUP (ORDER BY NEWS_DATE) AS "ニュースID"
FROM MARKET_NEWS
GROUP BY SENTIMENT
ORDER BY "件数" DESC;


### 2-3. AI_CLASSIFY（テキスト自動分類）

`AI_CLASSIFY(text, [categories])` でテキストを指定カテゴリに自動分類します。
ETFへの影響度別にニュースを振り分ける例を示します。

In [ ]:
-- ニュースを投資判断カテゴリに自動分類
SELECT
    NEWS_DATE   AS "日付",
    HEADLINE    AS "ヘッドライン",
    AI_CLASSIFY(
        LEFT(HEADLINE || ' ' || COALESCE(BODY, ''), 1500),
        ['買い材料', '様子見', '売り材料']
    ):labels[0]::VARCHAR AS "投資判断",
    RELEVANT_ETFS AS "関連ETF"
FROM MARKET_NEWS
ORDER BY NEWS_DATE DESC;


### 2-4. AI_EXTRACT 実践：月次レポート画像から情報抽出

`AI_EXTRACT(file, responseFormat)` で **画像ファイル（PNG/JPG/PDF）** に含まれる非構造化データから、指定した項目を構造化して抽出します。

実際の月次レポートの第1ページ画像（`2641_monthly_report_p1.png`）から、ファンド名・基準価額・リターンなどのKPIを自動抽出します。

> 💡 **ポイント**: 手作業でのデータ入力が不要になり、レポート更新のたびに自動でデータを最新化できます。


#### フラット化してテーブル形式で確認

上のクエリは JSON 形式の生データを返します。次のクエリでは各フィールドを列として展開し、表形式で確認します。


In [ ]:
SELECT
    AI_EXTRACT(
        file => TO_FILE('@ETF_AI_HANDSON_DB.AI.DOCUMENTS_STAGE', '2641_monthly_report_p1.png'),
        responseFormat => {
            'fund_name':             'ファンドの正式名称は何ですか？',
            'report_date':           'このレポートの基準日（年月日）は何ですか？',
            'nav_per_unit':          '基準価額（1口あたりの価格、単位：円）はいくらですか？',
            'total_nav_billion_yen': '純資産総額（単位：億円）はいくらですか？',
            'monthly_return_pct':    '月間騰落率（%）はいくらですか？',
            'ytd_return_pct':        '年初来騰落率（%）はいくらですか？',
            'since_inception_pct':   '設定来騰落率（%）はいくらですか？',
            'distribution_per_unit': '1口あたり分配金（円）はいくらですか？',
            'expense_ratio_pct':     '信託報酬率（実質コスト、年率%）はいくらですか？',
            'top10_holdings':        '組入上位10銘柄の銘柄名と組入比率は？'
        }
    ) AS "月次レポートから抽出されたデータ";


In [ ]:
-- 抽出結果をフラット化してテーブル形式で確認
WITH extracted AS (
    SELECT
        AI_EXTRACT(
            file => TO_FILE('@ETF_AI_HANDSON_DB.AI.DOCUMENTS_STAGE', '2641_monthly_report_p1.png'),
            responseFormat => {
                'fund_name':             'ファンドの正式名称は何ですか？',
                'report_date':           'このレポートの基準日（年月日）は何ですか？',
                'nav_per_unit':          '基準価額（1口あたりの価格、単位：円）はいくらですか？',
                'total_nav_billion_yen': '純資産総額（単位：億円）はいくらですか？',
                'monthly_return_pct':    '月間騰落率（%）はいくらですか？',
                'ytd_return_pct':        '年初来騰落率（%）はいくらですか？',
                'since_inception_pct':   '設定来騰落率（%）はいくらですか？',
                'expense_ratio_pct':     '信託報酬率（実質コスト、年率%）はいくらですか？'
            }
        ) AS result
)
SELECT
    result:response:fund_name::STRING             AS "ファンド名",
    result:response:report_date::STRING           AS "基準日",
    result:response:nav_per_unit::STRING          AS "基準価額",
    result:response:total_nav_billion_yen::STRING AS "純資産総額",
    result:response:monthly_return_pct::STRING    AS "月次リターン",
    result:response:ytd_return_pct::STRING        AS "年初来リターン",
    result:response:since_inception_pct::STRING   AS "設定来リターン",
    result:response:expense_ratio_pct::STRING     AS "実質コスト"
FROM extracted;


### 2-5. AI_AGG（複数テキストの統合サマリー生成）

`AI_AGG(text, question)` で複数行のテキストを集約しながら自然言語で質問に答えます。
全ニュースを横断して「今月の市場動向サマリー」を自動生成します。

In [ ]:
-- カテゴリ別にニュースをAI集約分析（GROUP BY × AI_AGG）
SELECT
    CATEGORY AS "カテゴリ",
    COUNT(*) AS "記事数",
    AI_AGG(
        HEADLINE || ': ' || BODY,
        'あなたはETFポートフォリオマネージャーのアシスタントです。
        以下のニュース群を分析し、次の形式で簡潔にレポートしてください：
        【トレンド】このカテゴリの主要な動向を1〜2文で
        【投資シグナル】ポジティブ材料とネガティブ材料を箇条書きで
        【注目ETF】言及頻度の高いETFコードとその理由'
    ) AS "AI分析レポート"
FROM MARKET_NEWS
WHERE NEWS_DATE >= '2026-01-01'
GROUP BY CATEGORY
ORDER BY COUNT(*) DESC;

## まとめ

Part 1 完了！Cortex AI Functions の基本操作を体験しました。

### 更新されたデータ

| テーブル | 操作 | 内容 |
|---|---|---|
| `MARKET_NEWS` | UPDATE | CORTEX.SENTIMENT によるセンチメントフラグ付与 |

### AI Functions の威力

- **AI_COMPLETE**: 2,000文字のニュース → 2行に圧縮、情報収集時間を 1/10 に
- **CORTEX.SENTIMENT**: ポジネガトーンを数値化し、重要ニュースを自動フィルタリング
- **AI_CLASSIFY**: 投資判断ラベルの自動付与、アナリストの分類作業を自動化
- **AI_EXTRACT**: PDF・画像 → 構造化データ、手入力不要
- **AI_AGG**: 大量ドキュメントの横断要約、朝のブリーフィング資料を自動生成

### よくある質問

**Q: PDF のファクトシートを直接処理できますか？**
A: `AI_PARSE_DOCUMENT` 関数でステージ上の PDF を直接テキスト化できます。
   その後チャンキングし、Cortex Search に登録します（Part 3 で体験）。

**Q: 日本語でも精度は高いですか？**
A: はい。Snowflake Cortex は日本語対応の LLM を使用しており、
   金融ドメインの日本語テキストでも高精度で動作します。

**次のステップ**: `part2_cortex_analyst.ipynb` で Semantic View と
Cortex Analyst（自然言語 to SQL）を体験してください。
